# Step 3 — AutoML: SageMaker Autopilot

Automatically explore ML algorithms and find the best baseline model.

In [ ]:
import boto3
import sagemaker
from sagemaker.automl.automl import AutoML

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
PREFIX = "symptom-data"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
role = sagemaker.get_execution_role()

In [ ]:
automl = AutoML(
    role=role,
    target_attribute_name="output_text",
    sagemaker_session=session,
    max_candidates=10,
    max_runtime_per_training_job_in_seconds=600,
    total_job_runtime_in_seconds=3600,
    mode="ENSEMBLING",
)

automl.fit(
    inputs=f"s3://{BUCKET}/{PREFIX}/processed/train_processed.csv",
    job_name="symptom-automl",
    wait=False,   # Runs ~1 hour in background
    logs=False
)
print("Autopilot job started — check AWS Console → SageMaker → AutoML jobs")

## View Results (run after job completes)

In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name=REGION)

response = sm_client.list_candidates_for_auto_ml_job(
    AutoMLJobName="symptom-automl",
    SortBy="FinalObjectiveMetricValue",
    SortOrder="Descending"
)

print(f"{'Candidate':<55} {'Status':<12} {'Accuracy'}")
print("-" * 80)
for c in response["Candidates"]:
    name   = c["CandidateName"]
    status = c["CandidateStatus"]
    metric = c.get("FinalAutoMLJobObjectiveMetric", {}).get("Value", "N/A")
    print(f"{name:<55} {status:<12} {metric}")

## Key Finding
- **LightGBM achieved 100% training accuracy** — looks perfect, but it's **overfitting**
- With only 853 samples, the model memorised the training data
- LightGBM uses TF-IDF features: no semantic understanding
- This motivates using Bio_ClinicalBERT for better generalisation